# Baseline linear regression model

## download the data



In [ ]:
import requests
import datetime

import pandas as pd
import matplotlib as plt
from tqdm import tqdm
import zipfile
import os

In [ ]:
files= [('202304','./data'),('202305','./data')]

for file, path in files:
    url=f'https://divvy-tripdata.s3.amazonaws.com/{file}-divvy-tripdata.zip'
    resp=requests.get(url, stream=True)
    zip_save_path = f'{path}/{file}.zip'

    os.makedirs(path, exist_ok=True)

    with open(zip_save_path,"wb") as handle:
        for data in tqdm(resp.iter_content(chunk_size=1024),
                         desc=f'{file}',
                         postfix=f"save to {zip_save_path}",
                         total=int(resp.headers["Content-Length"])):
            handle.write(data)

    with zipfile.ZipFile(zip_save_path, 'r') as zip_ref:
        zip_ref.extractall(path)
    
    os.remove(zip_save_path)

In [ ]:
data = pd.read_csv("data/raw/202304-divvy-tripdata.csv")


In [ ]:
data.head()

In [ ]:
data.describe()

In [ ]:
# missing data for started at or station name
data.isnull().sum()

These missing data for stations cannot be used, these are bikes that people leave in the street. We want to focus on usage at docks so we are to remove this data.

In [ ]:
# check for station names in both train and test sets
data.isnull().sum()

In [ ]:
#removing rows without station name
data=data.dropna(subset=['start_station_id','end_station_id'])



In [ ]:
data.isnull().sum() #no missing data left

In [ ]:
data.describe() #enough data to train my model

# Target engineering

I decided to create a target value that is the net usage of bikes per station per hour of the day.
This target will be called `net_usage`

## Define target variable
Target will be the number of rental at start station by hour


## Features

1 - Time features extracted from started at and ended at (day, day of the week)
2 - Rideable type (optional)
3 - Start station name
4 - End station name
    

In [ ]:
data.columns

In [ ]:
# extract hour and day of the week for started_at and ended_at
data['started_at'] = pd.to_datetime(data['started_at'])
data['ended_at'] = pd.to_datetime(data['ended_at'])

In [ ]:
data['started_hour'] = data['started_at'].dt.hour
data['started_day_of_week'] = data['started_at'].dt.day_name()

data['ended_hour'] = data['ended_at'].dt.hour
data['ended_day_of_week'] = data['ended_at'].dt.day_name()



In [ ]:
# dataset grouping features
rentals = data.groupby(['start_station_name', 'started_hour', 'started_day_of_week']).size().reset_index(name='average_rentals')

returns = data.groupby(['end_station_name', 'ended_hour', 'ended_day_of_week']).size().reset_index(name='average_returns')


In [ ]:
usage_data = pd.merge(rentals, returns, left_on=['start_station_name', 'started_hour', 'started_day_of_week'], right_on=['end_station_name', 'ended_hour', 'ended_day_of_week'], how='outer')


In [ ]:
usage_data['average_rentals'] = usage_data['average_rentals'].fillna(0)
usage_data['average_returns'] = usage_data['average_returns'].fillna(0)


In [ ]:
usage_data['net_usage'] = usage_data['average_rentals'] - usage_data['average_returns']


In [ ]:
usage_data.head()

In [ ]:
#extract station, hour and day of the week of either or start or end

def extract_station_info(df):

  df['station_name'] = df.apply(lambda row: row['end_station_name'] if not pd.isna(row['end_station_name']) else row['start_station_name'], axis=1)
  df['hour'] = df.apply(lambda row: row['ended_hour'] if not pd.isna(row['end_station_name']) else row['started_hour'], axis=1)
  df['day_of_week'] = df.apply(lambda row: row['ended_day_of_week'] if not pd.isna(row['end_station_name']) else row['started_day_of_week'], axis=1)

  return df[['net_usage', 'station_name', 'hour', 'day_of_week']]

In [ ]:
usage_data_2 = extract_station_info(usage_data.copy())  


In [ ]:
usage_data_2.shape

In [ ]:
usage_data_2.head()

# Model training


In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


In [ ]:
features = usage_data_2[['station_name', 'hour', 'day_of_week']]
target = usage_data_2['net_usage']

In [ ]:
#label features by type
num_features= ['hour']
cat_features = ['station_name', 'day_of_week']

In [ ]:
cat_features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)


In [ ]:
Standard_Scaler = StandardScaler()
num_scaled_train = pd.DataFrame(Standard_Scaler.fit_transform(X_train[num_features]), columns=['hour_scaled'])
num_scaled_test = pd.DataFrame(Standard_Scaler.transform(X_test[num_features]), columns=['hour_scaled'])


In [ ]:
num_scaled_train.index = X_train.index
num_scaled_test.index = X_test.index


In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore')
ohe_cols_train = ohe.fit_transform(X_train[cat_features])
ohe_cols_test = ohe.transform(X_test[cat_features])


In [ ]:
ohe_cols_train.toarray()

In [ ]:
ohe_cols_train = pd.DataFrame(ohe.fit_transform(X_train[cat_features]).toarray(), columns = ohe.get_feature_names_out(cat_features))
ohe_cols_test = pd.DataFrame(ohe.transform(X_test[cat_features]).toarray(), columns = ohe.get_feature_names_out(cat_features))

In [ ]:
ohe_cols_train.index = X_train.index
ohe_cols_test.index = X_test.index

In [ ]:
transformed_X_train = pd.concat([num_scaled_train, ohe_cols_train], axis=1)
transformed_X_test = pd.concat([num_scaled_test, ohe_cols_test], axis=1)

In [ ]:
transformed_X_train.head(20)

In [ ]:
model = LinearRegression()


In [ ]:
model.fit(transformed_X_train, y_train)


In [ ]:
predictions = model.predict(transformed_X_test)


In [ ]:
mse = mean_squared_error(y_test, predictions)
print(f'Mean Squared Error: {mse}')